In [23]:
import sys
sys.path.append("..")
from src.features.building_height import *

import fiona
import pandas as pd


In [5]:
#define terms and folders

las_folder = 'I:\Imagery\MassGIS_LAS_files'
mmc_town_names = ['Arlington', 'Boston', 'Braintree', 'Brookline', 'Cambridge', 'Chelsea', 
             'Everett', 'Malden', 'Medford', 'Melrose', 'Newton', 'Quincy', 'Revere', 
             'Somerville', 'Watertown', 'Winthrop']


from datetime import datetime


In [ ]:
#run again only if need be!
'''
#note that Revere was removed from this list since it was the test one 


for town_name in mmc_town_names:
    
    print(town_name + ' processing starting at ' + str(datetime.now()))


    #create las dataset 
    las_dataset = create_las_dataset(town_name = town_name, 
                                     las_folder=las_folder) 
    
    #create an ndsm raster
    ndsm_raster = create_ndsm_raster(town_name=town_name,
                                    las_dataset=las_dataset)
    
'''

In [ ]:
#only run again if necessary
'''
for town_name in mmc_town_names:
    
    print(town_name + ' processing starting at ' + str(datetime.now()))

    #RUN STORIES FUNCTION
    stories = make_stories_layer(town_name = town_name) '''

Arlington processing starting at 2025-06-02 22:52:12.699008
Boston processing starting at 2025-06-02 22:55:21.927890
Braintree processing starting at 2025-06-02 23:12:53.931945
Brookline processing starting at 2025-06-02 23:17:08.974920
Cambridge processing starting at 2025-06-02 23:21:04.097327
Chelsea processing starting at 2025-06-02 23:24:47.695129
Everett processing starting at 2025-06-02 23:27:40.497958
Malden processing starting at 2025-06-02 23:30:31.669468
Medford processing starting at 2025-06-02 23:33:38.942438
Melrose processing starting at 2025-06-02 23:37:38.401380
Newton processing starting at 2025-06-02 23:40:30.652731
Quincy processing starting at 2025-06-02 23:45:38.313339
Revere processing starting at 2025-06-02 23:51:17.108913
Somerville processing starting at 2025-06-02 23:54:31.911883
Watertown processing starting at 2025-06-02 23:57:36.523784
Winthrop processing starting at 2025-06-03 00:00:41.067282


In [24]:
cool_roofs_fp = r'K:\DataServices\Projects\Current_Projects\Climate_Change\MVP_MMC_CoolRoofs_MVP\Data\Analysis_Data\Data_Cool_Roofs\2_Output\MMC_Cool_Roofs.shp'
cool_roofs_gdf = gpd.read_file(cool_roofs_fp)

cool_roofs_gdf.head()

In [74]:
cool_roof_fields = ['STRUCT_ID', 'LOC_ID', 'flat_roof', 'SITE_ADDR_', 'CITY', 'LUC_Assign']
cool_roofs = cool_roofs_gdf[cool_roof_fields].drop_duplicates()

In [ ]:
# MERGE FOOTPRINT LAYERS, ADD 'STORIES' FIELDS # 

path = r"\\data-sync\public\DataServices\Projects\Current_Projects"
project_gdb = os.path.join(path, 'Neighborhood_Planning_and_Zoning\Zoning_Projects\Rightsizing_Zoning_2025\RightsizingZoning_2025.gdb')
mass_mainland_crs = "EPSG:26986"

#concatonate all of the roofprint layers in the gdb

layer_list = fiona.listlayers(project_gdb)

merged_gdf = gpd.GeoDataFrame(pd.concat([gpd.read_file(project_gdb, layer=layer_name) for layer_name in layer_list], 
                                        ignore_index=True), crs=mass_mainland_crs).drop_duplicates()



height_stats_fields = ['MAX',  'MEAN', 'RANGE', 'MEDIAN', 'PCT90', 'PCT75', 'PCT25']


#add 'stories' fields
for stats_field in height_stats_fields:
    field_name = stats_field + '_stories'
    merged_gdf[field_name] = (merged_gdf[stats_field] - 1) / 3.3 #subtract a meter as an (arbitrary) amount of raised-ness as an estimate for meters in a story


#export


In [ ]:
#merge to parcel data with cool roofs, only keep flat roofs field  
building_height_fields = ['MAX',  'MEAN', 'MEDIAN', 'PCT90', 'PCT75', 'PCT25', 
                          'MAX_stories', 'MEAN_stories', 'MEDIAN_stories', 'PCT75_stories', 'PCT25_stories',
                          'STRUCT_ID', 'geometry']

# join to cool roofs data, drop duplicates 

#add primary structure field
joined_roofprints = merged_gdf[building_height_fields].merge(cool_roofs, on='STRUCT_ID', how='inner').drop_duplicates()
joined_roofprints['roof_sqm'] = joined_roofprints['geometry'].area
joined_roofprints['primary_structure'] = 0

#look at largest structure per parcel
joined_roofprints.loc[joined_roofprints.groupby('LOC_ID')['roof_sqm'].idxmax(),'primary_structure'] = 1

#if condo, overwrite (all are "primray structures")
joined_roofprints.loc[(joined_roofprints['LUC_Assign'] == '102'), 'primary_structure'] = 1


joined_roofprints.to_file(project_gdb, layer=('00_merged_enriched_footprints'), driver='OpenFileGDB')

## TO DO: DEBUG: DUPLICATE STRUCT_ID ## 
## TO DO: debug - the initial join must have been a "within" which means large buildings that extend beyond the 
#   parcel boundary are not recognized as part of the parcel.
#   so it probably makes sense to re-join to parcels with better rules.

In [77]:
print('rows in joined_roofprints: ', len(joined_roofprints))
print('rows in cool_roofs: ', len(cool_roofs))
#print('rows in primary_structures: ', len(primary_structures))


print('unique LOC_ID in joined_roofprints: ', joined_roofprints['LOC_ID'].nunique())
print('unique LOC_ID in cool_roofs: ', cool_roofs['LOC_ID'].nunique())
#print('unique LOC_ID in primary_structures: ', primary_structures['LOC_ID'].nunique())


print('unique STRUCT_ID in joined_roofprints: ', joined_roofprints['STRUCT_ID'].nunique())
print('unique STRUCT_ID in cool_roofs: ', cool_roofs['STRUCT_ID'].nunique())
#print('unique STRUCT_ID in primary_structures: ', primary_structures['STRUCT_ID'].nunique())

rows in joined_roofprints:  328741
rows in cool_roofs:  327764
unique LOC_ID in joined_roofprints:  231126
unique LOC_ID in cool_roofs:  231126
unique STRUCT_ID in joined_roofprints:  327306
unique STRUCT_ID in cool_roofs:  327310


In [78]:
duplicates = joined_roofprints[joined_roofprints.duplicated(subset='STRUCT_ID')]
